In [ ]:
import os
from pathlib import Path

# Ensure relative paths resolve from the project root when running from notebooks/
if Path.cwd().name == "notebooks":
    os.chdir("..")

In [ ]:
import config

# Xanthan Rheology Plots

Viscosity and stress vs. shear rate for xanthan solutions at different concentrations,
with power-law (Ostwald–de Waele) fits on the stress–shear rate data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.optimize import curve_fit
import openpyxl

from klarity.viz import Colors

plt.rcParams.update(
    {
        "font.family": "serif",
        "font.size": 11,
        "axes.labelsize": 12,
        "axes.titlesize": 12,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "legend.fontsize": 9,
        "figure.dpi": 150,
    }
)

## Load data from Excel

In [ ]:
# ── Adjust this path to your Excel file location ──
EXCEL_PATH = config.RHEOLOGY_XLSX
OUT_PATH = "xanthan_plots/"

wb = openpyxl.load_workbook(EXCEL_PATH, data_only=True)
ws = wb["Sheet1"]

# Read all rows into a list of lists
raw = []
for row in ws.iter_rows(min_row=1, max_row=ws.max_row, values_only=True):
    raw.append(list(row))

# Layout (0-indexed columns):
# 0.125 wt% 1. Batch: Stress(MPa)=1, Stress(Pa)=2, Shear_rate=3, Viscosity=4
# 0.125 wt% 2. Batch: Stress(MPa)=7, Stress(Pa)=8, Shear_rate=9, Viscosity=10
# 0.25  wt% 1. Batch: Stress(MPa)=12, Stress(Pa)=13, Shear_rate=14, Viscosity=15
# 0.25  wt% 2. Batch: Stress(MPa)=18, Stress(Pa)=19, Shear_rate=20, Viscosity=21

data_rows = raw[2:]  # skip header rows


def extract_col(rows, col_idx):
    """Extract a numeric column, skipping None values."""
    vals = []
    for r in rows:
        v = r[col_idx]
        if v is not None:
            vals.append(float(v))
    return np.array(vals)


# Dataset dict: key = label, value = dict with shear_rate, viscosity, stress_pa
datasets = {
    "0.125 wt% 1. Batch": {
        "shear_rate": extract_col(data_rows, 3),
        "viscosity": extract_col(data_rows, 4),
        "stress_pa": extract_col(data_rows, 2),
    },
    "0.125 wt% 2. Batch": {
        "shear_rate": extract_col(data_rows, 9),
        "viscosity": extract_col(data_rows, 10),
        "stress_pa": extract_col(data_rows, 8),
    },
    "0.25 wt% 1. Batch": {
        "shear_rate": extract_col(data_rows, 14),
        "viscosity": extract_col(data_rows, 15),
        "stress_pa": extract_col(data_rows, 13),
    },
    "0.25 wt% 2. Batch": {
        "shear_rate": extract_col(data_rows, 20),
        "viscosity": extract_col(data_rows, 21),
        "stress_pa": extract_col(data_rows, 19),
    },
}

for label, d in datasets.items():
    print(
        f"{label}: {len(d['shear_rate'])} data points, "
        f"shear rate range [{d['shear_rate'].min():.1f}, {d['shear_rate'].max():.1f}] 1/s"
    )

## Power-law (Ostwald–de Waele) fit

The power-law model relates shear stress $\tau$ to shear rate $\dot{\gamma}$:

$$\tau = K \, \dot{\gamma}^{\,n}$$

where $K$ is the consistency index and $n$ is the flow behaviour index.

In [ ]:
def power_law(gamma_dot, K, n):
    """Ostwald-de Waele: tau = K * gamma_dot^n"""
    return K * np.power(gamma_dot, n)


# Fit each dataset
fit_params = {}
for label, d in datasets.items():
    popt, pcov = curve_fit(power_law, d["shear_rate"], d["stress_pa"], p0=[0.1, 0.5])
    K, n = popt
    fit_params[label] = {"K": K, "n": n, "pcov": pcov}
    print(f"{label}:  K = {K:.4f} Pa·s^n,  n = {n:.4f}")

## Plot styling

In [ ]:
# Colour and marker mapping matching the original Excel plots
style = {
    "0.125 wt% 1. Batch": {"color": Colors.orange, "marker": "s", "label": "0.125 wt%, 1. Batch"},
    "0.125 wt% 2. Batch": {"color": Colors.orange, "marker": "o", "label": "0.125 wt%, 2. Batch"},
    "0.25 wt% 1. Batch": {"color": Colors.blue, "marker": "s", "label": "0.25  wt%, 1. Batch"},
    "0.25 wt% 2. Batch": {"color": Colors.blue, "marker": "o", "label": "0.25  wt%, 2. Batch"},
}

# Fit-line colours: slightly different shade per batch for visual distinction
fit_line_style = {
    "0.125 wt% 1. Batch": {"color": Colors.orange, "ls": "-"},
    "0.125 wt% 2. Batch": {"color": Colors.orange, "ls": "-"},
    "0.25 wt% 1. Batch": {"color": Colors.blue, "ls": "-"},
    "0.25 wt% 2. Batch": {"color": Colors.blue, "ls": "-"},
}

## Figure: Viscosity and Stress vs. Shear Rate (side by side)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(6.7, 3))

# ── Left panel: Viscosity vs Shear Rate ──
for label, d in datasets.items():
    s = style[label]
    ax1.plot(
        d["shear_rate"],
        d["viscosity"],
        marker=s["marker"],
        color=s["color"],
        linestyle="none",
        markersize=5,
        label=s["label"],
    )

ax1.set_xlabel(r"Shear rate [s$^{-1}$]")
ax1.set_ylabel("$\mu$ [Pa s]")
ax1.set_xlim(left=0)
ax1.set_ylim(bottom=0)
ax1.legend(frameon=False, fancybox=False, edgecolor="black")

# ── Right panel: Stress vs Shear Rate with power-law fits ──
gamma_fit = np.linspace(0.5, 340, 500)

for label, d in datasets.items():
    s = style[label]
    ax2.plot(
        d["shear_rate"],
        d["stress_pa"],
        marker=s["marker"],
        color=s["color"],
        linestyle="none",
        markersize=5,
        label=s["label"],
    )
    # Overlay fit curve
    fs = fit_line_style[label]
    fp = fit_params[label]
    ax2.plot(
        gamma_fit,
        power_law(gamma_fit, fp["K"], fp["n"]),
        color=fs["color"],
        ls=fs["ls"],
        lw=1,
    )

ax2.set_xlabel(r"Shear rate [s$^{-1}$]")
ax2.set_ylabel("Stress [Pa]")
ax2.set_xlim(left=0)
ax2.set_ylim(bottom=0)
# ax2.legend(frameon=False, fancybox=False, edgecolor="black")

fig.tight_layout()
fig.savefig(OUT_PATH + "xanthan_rheology_combined.png", dpi=1200, bbox_inches="tight")
fig.savefig(OUT_PATH + "xanthan_rheology_combined.pdf", bbox_inches="tight")
plt.show()
print("Saved: xanthan_rheology_combined.png / .pdf")

## Individual plots (if needed for paper figures)

In [ ]:
# ── Viscosity plot only ──
fig_v, ax_v = plt.subplots(figsize=(5.5, 4.2))

for label, d in datasets.items():
    s = style[label]
    ax_v.plot(
        d["shear_rate"],
        d["viscosity"],
        marker=s["marker"],
        color=s["color"],
        linestyle="none",
        markersize=5,
        label=s["label"],
    )

ax_v.set_xlabel(r"Shear rate [s$^{-1}$]")
ax_v.set_ylabel("$\mu$ [Pa s]")
ax_v.set_xlim(left=0)
ax_v.set_ylim(bottom=0)
ax_v.legend(frameon=False, fancybox=False, edgecolor="black")
fig_v.tight_layout()
fig_v.savefig(OUT_PATH + "xanthan_viscosity.png", dpi=1200, bbox_inches="tight")
fig_v.savefig(OUT_PATH + "xanthan_viscosity.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# ── Stress plot only ──
fig_s, ax_s = plt.subplots(figsize=(5.5, 4.2))

gamma_fit = np.linspace(0.5, 340, 500)

for label, d in datasets.items():
    s = style[label]
    ax_s.plot(
        d["shear_rate"],
        d["stress_pa"],
        marker=s["marker"],
        color=s["color"],
        linestyle="none",
        markersize=5,
        label=s["label"],
    )
    fs = fit_line_style[label]
    fp = fit_params[label]
    ax_s.plot(
        gamma_fit,
        power_law(gamma_fit, fp["K"], fp["n"]),
        color=fs["color"],
        ls=fs["ls"],
        lw=1,
    )

ax_s.set_xlabel(r"Shear rate [s$^{-1}$]")
ax_s.set_ylabel("Stress [Pa]")
ax_s.set_xlim(left=0)
ax_s.set_ylim(bottom=0)
ax_s.legend(frameon=False, fancybox=False, edgecolor="black")
fig_s.tight_layout()
fig_s.savefig(OUT_PATH + "xanthan_stress.png", dpi=1200, bbox_inches="tight")
fig_s.savefig(OUT_PATH + "xanthan_stress.pdf", bbox_inches="tight")
plt.show()

## Summary table of power-law fit parameters

In [ ]:
print(f"{'Dataset':<28s}  {'K (Pa·s^n)':>12s}  {'n':>8s}")
print("-" * 52)
for label, fp in fit_params.items():
    print(f"{label:<28s}  {fp['K']:12.6f}  {fp['n']:8.4f}")